In [1]:
pip install tensorflow scikit-learn pillow matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

BASE_DIR = "./power-pole-dataset"
CATEGORIES = ['aman', 'bahaya']

print(f"[DEBUG] Mengecek keberadaan folder base: {os.path.abspath(BASE_DIR)}")

# KONFIGURASI AUGMENTASI
datagen = ImageDataGenerator(
    horizontal_flip=True,                
    brightness_range=[0.6, 1.4],         
    fill_mode='nearest'                  
)

def eksekusi_upsampling():
    TARGET_TOTAL = 1000 
    info_data = {}
    
    for cat in CATEGORIES:
        cat_path = os.path.join(BASE_DIR, cat)
        print(f"Mengecek folder kelas: {os.path.abspath(cat_path)}")
        
        if os.path.exists(cat_path):
            # Ambil semua file gambar
            all_imgs = [f for f in os.listdir(cat_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            info_data[cat] = {
                'path': cat_path,
                'files': all_imgs,
                'total': len(all_imgs)
            }
            print(f"Kelas '{cat}' mendeteksi: {len(all_imgs)} gambar asli.")
        else:
            print(f"Folder {cat_path} TIDAK DITEMUKAN.")
            return

    print(f"\nTarget penyeimbangan data DIKUNCI pada: {TARGET_TOTAL} gambar per kelas.\n")

    for cat in CATEGORIES:
        total_sekarang = info_data[cat]['total']
        
        if total_sekarang >= TARGET_TOTAL:
            print(f"Kelas '{cat}' sudah memiliki {total_sekarang} gambar.")
            continue
            
        kekurangan = TARGET_TOTAL - total_sekarang
        print(f"Kelas '{cat}' kekurangan {kekurangan} gambar.")
        
        img_list = info_data[cat]['files']
        folder_target = info_data[cat]['path']
        counter = 0
        
        while counter < kekurangan:
            img_acak = np.random.choice(img_list)
            img_path = os.path.join(folder_target, img_acak)
            
            try:
                img = load_img(img_path)
                x = img_to_array(img)
                x = x.reshape((1,) + x.shape)
                
                for batch in datagen.flow(x, batch_size=1, 
                                          save_to_dir=folder_target, 
                                          save_prefix=f"aug_{counter}", 
                                          save_format="jpg"):
                    counter += 1
                    break 
            except Exception as e:
                print(f"Gagal augmentasi pada file {img_acak}: {e}")
                
        print(f"Kelas '{cat}' selesai di-upsampling! Total sekarang: {TARGET_TOTAL} gambar.\n")
        
    print(f"Selesai! Semua kelas sekarang masing-masing berisi {TARGET_TOTAL} gambar.")

eksekusi_upsampling()

[DEBUG] Mengecek keberadaan folder base: e:\ABEL\Dataset Abel\new dataset\power-pole-dataset
Mengecek folder kelas: e:\ABEL\Dataset Abel\new dataset\power-pole-dataset\aman
Kelas 'aman' mendeteksi: 190 gambar asli.
Mengecek folder kelas: e:\ABEL\Dataset Abel\new dataset\power-pole-dataset\bahaya
Kelas 'bahaya' mendeteksi: 516 gambar asli.

Target penyeimbangan data DIKUNCI pada: 1000 gambar per kelas.

Kelas 'aman' kekurangan 810 gambar.
Kelas 'aman' selesai di-upsampling! Total sekarang: 1000 gambar.

Kelas 'bahaya' kekurangan 484 gambar.
Kelas 'bahaya' selesai di-upsampling! Total sekarang: 1000 gambar.

Selesai! Semua kelas sekarang masing-masing berisi 1000 gambar.


In [2]:
import shutil
import random
from sklearn.model_selection import train_test_split

NILAI_SEED = 42
random.seed(NILAI_SEED)

SRC_DIR = "./power-pole-dataset"
CATEGORIES = ['aman', 'bahaya']
DEST_DIR = "./power-pole-dataset-70"

#PROSES SPLITTING DATA (70:20:10) DENGAN SEED

def splitting_data():
    print(f"Memulai proses splitting data (Kunci Seed: {NILAI_SEED})")
    print(f"Dari: '{SRC_DIR}' ke '{DEST_DIR}'")

    for cat in CATEGORIES:
        cat_src_path = os.path.join(SRC_DIR, cat)
        if not os.path.exists(cat_src_path):
            print(f"Folder kelas {cat_src_path} tidak ditemukan. Skip.")
            continue

        all_imgs = sorted([f for f in os.listdir(cat_src_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        total_gambar = len(all_imgs)
        print(f"Memproses Kelas '{cat}' (Total: {total_gambar} gambar):")

        #PERHITUNGAN RASIO (70:20:10) DENGAN RANDOM STATE
        train_val_imgs, test_imgs = train_test_split(
            all_imgs, 
            test_size=0.10, 
            random_state=NILAI_SEED, 
            shuffle=True
        )

        val_size_relative = 0.20 / 0.90
        train_imgs, val_imgs = train_test_split(
            train_val_imgs, 
            test_size=val_size_relative, 
            random_state=NILAI_SEED, 
            shuffle=True
        )

        print(f"  -> Train (70%)     : {len(train_imgs)} gambar")
        print(f"  -> Validation (20%): {len(val_imgs)} gambar")
        print(f"  -> Test (10%)      : {len(test_imgs)} gambar")

        # simpan
        splits = {
            'train': train_imgs,
            'valid': val_imgs,
            'test': test_imgs
        }

        for split_name, img_list in splits.items():
            target_folder = os.path.join(DEST_DIR, split_name, cat)
            os.makedirs(target_folder, exist_ok=True)

            for img_name in img_list:
                src_file = os.path.join(cat_src_path, img_name)
                dest_file = os.path.join(target_folder, img_name)
                shutil.copy(src_file, dest_file)

        print(f"Kelas '{cat}' sukses di-split secara konsisten.\n")

    print(f"Selesai! Folder '{DEST_DIR}' berhasil dikunci dengan seed.")

splitting_data()

Memulai proses splitting data (Kunci Seed: 42)
Dari: './power-pole-dataset' ke './power-pole-dataset-70'
Memproses Kelas 'aman' (Total: 1000 gambar):
  -> Train (70%)     : 700 gambar
  -> Validation (20%): 200 gambar
  -> Test (10%)      : 100 gambar
Kelas 'aman' sukses di-split secara konsisten.

Memproses Kelas 'bahaya' (Total: 1000 gambar):
  -> Train (70%)     : 700 gambar
  -> Validation (20%): 200 gambar
  -> Test (10%)      : 100 gambar
Kelas 'bahaya' sukses di-split secara konsisten.

Selesai! Folder './power-pole-dataset-70' berhasil dikunci dengan seed.
